## Experiment - LLM (GPT2) - WikiText - Figures

- This notebook generates the manuscript metric figures for the LLM (GPT2) Wikitext CRT experiment. 

In [ ]:
# check if helvetica is available
import matplotlib.font_manager as fm
print([f.name for f in fm.fontManager.ttflist if "Helvetica" in f.name])

## Add global plots + combined plots 

In [ ]:
#!/usr/bin/env python3
import json
import math
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import numpy as np
from cycler import cycler

# Set font to Arial (sans-serif)
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]
# Set font sizes for publication
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 20
mpl.rcParams["axes.labelsize"] = 18
mpl.rcParams["legend.fontsize"] = 14
mpl.rcParams["legend.title_fontsize"] = 16
# Axis tick label sizes
mpl.rcParams["xtick.labelsize"] = 16
mpl.rcParams["ytick.labelsize"] = 16

# Viridis color cycle
viridis_colors = plt.cm.viridis(np.linspace(0.15, 0.85, 3))
mpl.rcParams["axes.prop_cycle"] = cycler(color=viridis_colors)



def read_history_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    if not rows:
        raise RuntimeError(f"No rows found in {path}")
    df = pd.DataFrame(rows)

    # Ensure numeric where possible
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = pd.to_numeric(df[c], errors="ignore")

    if "iter" not in df.columns:
        raise RuntimeError("Expected an 'iter' field in history.jsonl rows.")
    df = df.sort_values("iter").reset_index(drop=True)
    return df


def savefig(outpath: Path, global_dir: Path = None, run_name: str = None):
    outpath.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()

    # Save to original location
    plt.savefig(outpath, dpi=300)

    # Also save to global directory with run-prefixed filename
    if global_dir is not None and run_name is not None:
        global_dir.mkdir(parents=True, exist_ok=True)

        # Normal global version
        global_outpath = global_dir / f"{run_name}_{outpath.name}"
        plt.savefig(global_outpath, dpi=300)

        # Clean global version: no title, no legend, no axis labels
        ax = plt.gca()
        old_title = ax.get_title()
        old_xlabel = ax.get_xlabel()
        old_ylabel = ax.get_ylabel()
        legend = ax.get_legend()

        ax.set_title("")
        ax.set_xlabel("")
        ax.set_ylabel("")
        if legend is not None:
            legend.remove()

        clean_outpath = global_dir / f"{run_name}_{outpath.stem}_clean{outpath.suffix}"
        plt.savefig(clean_outpath, dpi=300)

        # restore
        ax.set_title(old_title)
        ax.set_xlabel(old_xlabel)
        ax.set_ylabel(old_ylabel)

    plt.close()


def savefig_single(outpath: Path):
    outpath.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()

    # Normal version
    plt.savefig(outpath, dpi=300)

    # Clean version: no title, no legend, no axis labels
    ax = plt.gca()
    old_title = ax.get_title()
    old_xlabel = ax.get_xlabel()
    old_ylabel = ax.get_ylabel()
    legend = ax.get_legend()

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    if legend is not None:
        legend.remove()

    clean_outpath = outpath.parent / f"{outpath.stem}_clean{outpath.suffix}"
    plt.savefig(clean_outpath, dpi=300)

    # restore
    ax.set_title(old_title)
    ax.set_xlabel(old_xlabel)
    ax.set_ylabel(old_ylabel)

    plt.close()


def make_figure_label(name: str) -> str:
    if name == "iter":
        return "Iteration"
    return name


def make_legend_label(name: str) -> str:
    mapping = {
        "entropy_bits": "Entropy (bits)",
        "entropy_nats": "Entropy (nats)",
        "distinct_1": "Distinct-1",
        "distinct_2": "Distinct-2",
        "distinct_3": "Distinct-3",
        "m_real": "Real blocks",
        "m_synth": "Synthetic blocks",
        "test_ppl": "Test Perplexity",
        "log_test_ppl": "log(Test Perplexity)",
        "train_wall_sec": "Train wall time",
        "eval_wall_sec": "Eval wall time",
        "sample_wall_sec": "Sample wall time",
        "iter_wall_sec": "Iteration wall time",
        "total_gen_sec": "Total generation seconds",
        "approx_blocks_per_sec": "Blocks/sec",
        "approx_tokens_per_sec": "Tokens/sec",
    }
    return mapping.get(name, name)


def extract_alpha_value(run_name: str):
    """
    Extract numeric alpha from a run name like:
      alpha=0.1_llm_recursive_mix
    Returns float if found, else None.
    """
    if "_" in run_name:
        base = run_name.split("_")[0]
    else:
        base = run_name

    if base.startswith("alpha="):
        try:
            return float(base.split("=", 1)[1])
        except ValueError:
            return None
    return None


def pretty_run_label(run_name: str) -> str:
    """
    Convert:
      alpha=0.5_llm_recursive_mix -> r"$\\alpha=0.5$"
    while preserving the alpha=0.0 style formatting.
    """
    if "_" in run_name:
        base = run_name.split("_")[0]
    else:
        base = run_name

    alpha_val = extract_alpha_value(run_name)
    if alpha_val is not None:
        return rf"$\alpha={alpha_val:.1f}$"

    return base


def sanitize_ylim_for_scale(ylim, log_scale: bool):
    if ylim is None:
        return None
    lower, upper = ylim
    if log_scale and lower is not None and lower <= 0:
        print(f"Invalid ylim {ylim} for log scale. Adjusting lower bound to a small positive value.")
        lower = 1e-3
    return (lower, upper)


def plot_line(
    df: pd.DataFrame,
    x: str,
    y: str,
    title: str,
    outpath: Path,
    ylabel: str = "",
    global_dir: Path = None,
    run_name: str = None,
    log_scale: bool = False,
    ylim=None,
):
    if y not in df.columns:
        print(f"Skipping {y} (not present).")
        return

    plot_df = df[[x, y]].copy()
    plot_df[y] = pd.to_numeric(plot_df[y], errors="coerce")

    if log_scale:
        plot_df.loc[plot_df[y] <= 0, y] = float("nan")
        if plot_df[y].dropna().empty:
            print(f"Skipping {y} with log scale (no positive values).")
            return

    plt.figure()
    plt.plot(plot_df[x], plot_df[y], marker="o", linewidth=1)
    plt.xlabel(make_figure_label(x))
    plt.ylabel(ylabel if ylabel else make_legend_label(y))
    plt.title(title)

    if log_scale:
        plt.yscale("log")

    ylim = sanitize_ylim_for_scale(ylim, log_scale)
    if ylim is not None:
        plt.ylim(ylim)

    savefig(outpath, global_dir=global_dir, run_name=run_name)


def plot_multi(
    df: pd.DataFrame,
    x: str,
    ys,
    title: str,
    outpath: Path,
    ylabel: str = "",
    global_dir: Path = None,
    run_name: str = None,
    log_scale: bool = False,
    ylim=None,
):
    ys_present = [y for y in ys if y in df.columns]
    if not ys_present:
        print(f"Skipping {title} (none of {ys} present).")
        return

    plt.figure()
    plotted_any = False

    for y in ys_present:
        yvals = pd.to_numeric(df[y], errors="coerce").copy()

        if log_scale:
            yvals[yvals <= 0] = float("nan")
            if yvals.dropna().empty:
                print(f"Skipping {y} in {title} with log scale (no positive values).")
                continue

        plt.plot(df[x], yvals, marker="o", linewidth=1, label=make_legend_label(y))
        plotted_any = True

    if not plotted_any:
        plt.close()
        print(f"Skipping {title} (nothing plottable after log filtering).")
        return

    plt.xlabel(make_figure_label(x))
    plt.ylabel(ylabel if ylabel else "")
    plt.title(title)
    plt.legend()

    if log_scale:
        plt.yscale("log")

    ylim = sanitize_ylim_for_scale(ylim, log_scale)
    if ylim is not None:
        plt.ylim(ylim)

    savefig(outpath, global_dir=global_dir, run_name=run_name)


def plot_compare_runs(
    run_dfs: dict,
    metric: str,
    title: str,
    outpath: Path,
    ylabel: str = "",
    x: str = "iter",
    log_scale: bool = False,
    ylim=None,
    show_y = False,
    show_legend = False,
):
    plotted_any = False
    plt.figure()

    # Sort runs numerically by alpha when possible
    sorted_items = sorted(
        run_dfs.items(),
        key=lambda kv: (
            extract_alpha_value(kv[0]) is None,
            float("inf") if extract_alpha_value(kv[0]) is None else extract_alpha_value(kv[0]),
            kv[0],
        ),
    )

    for run_name, df in sorted_items:
        if metric not in df.columns:
            print(f"Skipping {metric} for {run_name} (not present).")
            continue
        if x not in df.columns:
            print(f"Skipping {run_name}: missing x column {x}.")
            continue

        yvals = pd.to_numeric(df[metric], errors="coerce").copy()

        if log_scale:
            yvals[yvals <= 0] = float("nan")
            if yvals.dropna().empty:
                print(f"Skipping {metric} for {run_name} with log scale (no positive values).")
                continue

        plt.plot(
            df[x],
            yvals,
            marker="o",
            linewidth=1,
            label=pretty_run_label(run_name),
        )
        plotted_any = True

    if not plotted_any:
        plt.close()
        print(f"Skipping comparison plot {title} (metric {metric} missing in all runs).")
        return

    plt.xlabel(make_figure_label(x))
    if show_y:
        plt.ylabel(ylabel if ylabel else make_legend_label(metric))
    plt.title(title)
    if show_legend:
        plt.legend()
        plt.legend(title="Legend")

    if log_scale:
        plt.yscale("log")

    ylim = sanitize_ylim_for_scale(ylim, log_scale)
    if ylim is not None:
        plt.ylim(ylim)

    savefig_single(outpath)


def add_log_perplexity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "test_ppl" in df.columns:
        df["log_test_ppl"] = df["test_ppl"].apply(
            lambda v: math.log(v) if isinstance(v, (int, float)) and pd.notna(v) and v > 0 else float("nan")
        )
    return df


def main():
    # --------------------------------------------------
    # Define the runs you want to compare
    # Edit this list as needed
    # --------------------------------------------------
    run_dirs = [
        Path("runs/alpha=0.0_llm_recursive_mix"),
        Path("runs/alpha=0.1_llm_recursive_mix"),
        Path("runs/alpha=1.0_llm_recursive_mix"),
    ]

    # --------------------------------------------------
    # Optional log scaling
    # Put the metric names here if you want them on log scale
    # --------------------------------------------------
    log_metrics = {
        # "test_ppl",
        # "entropy_bits",
        # "distinct_1",
        # "distinct_2",
        # "distinct_3",
        # "m_real",
        # "m_synth",
        # "train_wall_sec",
        # "eval_wall_sec",
        # "sample_wall_sec",
        # "iter_wall_sec",
        # "total_gen_sec",
        # "approx_blocks_per_sec",
        # "approx_tokens_per_sec",
    }

    global_plots_dir = Path("plots_global")
    compare_plots_dir = global_plots_dir / "comparisons"

    global_plots_dir.mkdir(parents=True, exist_ok=True)
    compare_plots_dir.mkdir(parents=True, exist_ok=True)

    run_dfs = {}

    # --------------------------------------------------
    # Existing behavior: save each run's plots locally
    # and also copy them into plots_global with prefixed names
    # --------------------------------------------------
    for run_dir in run_dirs:
        run_name = run_dir.name
        plots_dir = run_dir / "plots"
        plots_dir.mkdir(parents=True, exist_ok=True)

        hist_path = run_dir / "history.jsonl"
        if not hist_path.exists():
            print(f"Skipping {run_name}: missing {hist_path}")
            continue

        df = read_history_jsonl(hist_path)
        df = add_log_perplexity(df)
        run_dfs[run_name] = df

        # Save CSV locally + globally
        df.to_csv(plots_dir / "history.csv", index=False)
        df.to_csv(global_plots_dir / f"{run_name}_history.csv", index=False)

        # --- Perplexity ---
        plot_line(
            df, "iter", "test_ppl",
            "Test Perplexity",
            plots_dir / "plot_test_ppl.png",
            ylabel="Perplexity",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("test_ppl" in log_metrics),
        )

        # --- log perplexity ---
        plot_line(
            df, "iter", "log_test_ppl",
            "log(Test Perplexity)",
            plots_dir / "plot_log_test_ppl.png",
            ylabel="log(Perplexity)",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=False,  # already transformed
        )

        # --- Entropy ---
        plot_multi(
            df, "iter", ["entropy_bits"],
            "Predictive Entropy",
            plots_dir / "plot_entropy.png",
            ylabel="Entropy",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("entropy_bits" in log_metrics),
        )

        # --- Distinct-n ---
        plot_multi(
            df, "iter", ["distinct_1", "distinct_2", "distinct_3"],
            "Distinct-n",
            plots_dir / "plot_distinct_n.png",
            ylabel="Distinct-n",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=any(m in log_metrics for m in ["distinct_1", "distinct_2", "distinct_3"]),
            ylim=(0, 1),
        )

        # --- Dataset composition ---
        plot_multi(
            df, "iter", ["m_real", "m_synth"],
            "Real vs synthetic blocks per iteration",
            run_dir / "plot_real_vs_synth.png",
            ylabel="blocks",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=any(m in log_metrics for m in ["m_real", "m_synth"]),
        )

        # --- Timing ---
        plot_multi(
            df, "iter", ["train_wall_sec", "eval_wall_sec", "sample_wall_sec", "iter_wall_sec"],
            "Wall-clock timing vs iteration",
            plots_dir / "plot_wall_times.png",
            ylabel="seconds",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=any(m in log_metrics for m in ["train_wall_sec", "eval_wall_sec", "sample_wall_sec", "iter_wall_sec"]),
        )

        # --- Synthetic timing fields ---
        plot_line(
            df, "iter", "total_gen_sec",
            "Synthetic generation total_gen_sec vs iteration (when generated)",
            plots_dir / "plot_synth_total_gen_sec.png",
            ylabel="seconds",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("total_gen_sec" in log_metrics),
        )
        plot_line(
            df, "iter", "approx_blocks_per_sec",
            "Synthetic generation throughput (blocks/sec) vs iteration",
            plots_dir / "plot_synth_blocks_per_sec.png",
            ylabel="blocks/sec",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("approx_blocks_per_sec" in log_metrics),
        )
        plot_line(
            df, "iter", "approx_tokens_per_sec",
            "Synthetic generation throughput (tokens/sec) vs iteration",
            plots_dir / "plot_synth_tokens_per_sec.png",
            ylabel="tokens/sec",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("approx_tokens_per_sec" in log_metrics),
        )

        print(f"Saved per-run plots for: {run_name}")

    # --------------------------------------------------
    # Comparison plots across runs
    # Saved into plots_global/comparisons/
    # --------------------------------------------------
    if not run_dfs:
        print("No valid runs found. No comparison plots created.")
        return

    comparison_specs = [
        ("test_ppl", "Test Perplexity", "compare_test_ppl.png", "Perplexity"),
        ("log_test_ppl", "(log) Test Perplexity", "compare_log_test_ppl.png", "log(Perplexity)"),
        ("entropy_bits", "Entropy (bits)", "compare_entropy_bits.png", "Entropy (bits)"),
        ("distinct_1", "Distinct-1", "compare_distinct_1.png", "Distinct-1"),
        ("distinct_2", "Distinct-2", "compare_distinct_2.png", "Distinct-2"),
        ("distinct_3", "Distinct-3", "compare_distinct_3.png", "Distinct-3"),
        ("m_real", "Real Blocks vs Iteration Across Runs", "compare_m_real.png", "blocks"),
        ("m_synth", "Synthetic Blocks vs Iteration Across Runs", "compare_m_synth.png", "blocks"),
        ("train_wall_sec", "Train Wall Time vs Iteration Across Runs", "compare_train_wall_sec.png", "seconds"),
        ("eval_wall_sec", "Eval Wall Time vs Iteration Across Runs", "compare_eval_wall_sec.png", "seconds"),
        ("sample_wall_sec", "Sample Wall Time vs Iteration Across Runs", "compare_sample_wall_sec.png", "seconds"),
        ("iter_wall_sec", "Iteration Wall Time vs Iteration Across Runs", "compare_iter_wall_sec.png", "seconds"),
        ("total_gen_sec", "Total Generation Time vs Iteration Across Runs", "compare_total_gen_sec.png", "seconds"),
        ("approx_blocks_per_sec", "Generation Throughput (blocks/sec) Across Runs", "compare_blocks_per_sec.png", "blocks/sec"),
        ("approx_tokens_per_sec", "Generation Throughput (tokens/sec) Across Runs", "compare_tokens_per_sec.png", "tokens/sec"),
    ]

    for metric, title, filename, ylabel in comparison_specs:
        if metric in {"distinct_1", "distinct_2", "distinct_3"}:
            ylim = (0, 1)
        elif metric in {"entropy_bits"}:
            ylim = (1, None)
        elif metric in {"log_test_ppl"}:
            ylim = (2.5, None)
        else:
            ylim = None

        plot_compare_runs(
            run_dfs=run_dfs,
            metric=metric,
            title=title,
            outpath=compare_plots_dir / filename,
            ylabel=ylabel,
            log_scale=(metric in log_metrics),
            ylim=ylim,
        )

    print("Saved global copies to:", global_plots_dir.resolve())
    print("Saved comparison plots to:", compare_plots_dir.resolve())


if __name__ == "__main__":
    main()

## All titles removed
- also fixed the tick marks to be 1 decimal place

In [ ]:
#!/usr/bin/env python3
import json
import math
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import numpy as np
from cycler import cycler
from matplotlib.ticker import FormatStrFormatter


# -----------------------------
# GLOBAL STYLE
# -----------------------------

# Set font to Arial (sans-serif)
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]

# Set font sizes for publication
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 20
mpl.rcParams["axes.labelsize"] = 18
mpl.rcParams["legend.fontsize"] = 14
mpl.rcParams["legend.title_fontsize"] = 16

# Axis tick label sizes
mpl.rcParams["xtick.labelsize"] = 16
mpl.rcParams["ytick.labelsize"] = 16

# Viridis color cycle
viridis_colors = plt.cm.viridis(np.linspace(0.15, 0.85, 3))
mpl.rcParams["axes.prop_cycle"] = cycler(color=viridis_colors)


def format_y_axis_1_decimal(log_scale: bool = False):
    """
    Force y-axis tick labels to 1 decimal place.

    For normal axes:
        1 -> 1.0
        0 -> 0.0
        2.5 -> 2.5

    For log-scale axes, we skip this by default because fixed decimal
    formatting can make log ticks look strange.
    """
    if log_scale:
        return

    ax = plt.gca()
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))


def read_history_jsonl(path: Path) -> pd.DataFrame:
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    if not rows:
        raise RuntimeError(f"No rows found in {path}")

    df = pd.DataFrame(rows)

    # Ensure numeric where possible
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = pd.to_numeric(df[c], errors="ignore")

    if "iter" not in df.columns:
        raise RuntimeError("Expected an 'iter' field in history.jsonl rows.")

    df = df.sort_values("iter").reset_index(drop=True)

    return df


def savefig(outpath: Path, global_dir: Path = None, run_name: str = None):
    outpath.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()

    # Save to original location
    plt.savefig(outpath, dpi=300)

    # Also save to global directory with run-prefixed filename
    if global_dir is not None and run_name is not None:
        global_dir.mkdir(parents=True, exist_ok=True)

        # Normal global version
        global_outpath = global_dir / f"{run_name}_{outpath.name}"
        plt.savefig(global_outpath, dpi=300)

        # Clean global version: no title, no legend, no axis labels
        ax = plt.gca()
        old_title = ax.get_title()
        old_xlabel = ax.get_xlabel()
        old_ylabel = ax.get_ylabel()
        legend = ax.get_legend()

        ax.set_title("")
        ax.set_xlabel("")
        ax.set_ylabel("")

        if legend is not None:
            legend.remove()

        clean_outpath = global_dir / f"{run_name}_{outpath.stem}_clean{outpath.suffix}"
        plt.savefig(clean_outpath, dpi=300)

        # Restore
        ax.set_title(old_title)
        ax.set_xlabel(old_xlabel)
        ax.set_ylabel(old_ylabel)

    plt.close()


def savefig_single(outpath: Path):
    outpath.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()

    # Normal version
    plt.savefig(outpath, dpi=300)

    # Clean version: no title, no legend, no axis labels
    ax = plt.gca()
    old_title = ax.get_title()
    old_xlabel = ax.get_xlabel()
    old_ylabel = ax.get_ylabel()
    legend = ax.get_legend()

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")

    if legend is not None:
        legend.remove()

    clean_outpath = outpath.parent / f"{outpath.stem}_clean{outpath.suffix}"
    plt.savefig(clean_outpath, dpi=300)

    # Restore
    ax.set_title(old_title)
    ax.set_xlabel(old_xlabel)
    ax.set_ylabel(old_ylabel)

    plt.close()


def make_figure_label(name: str) -> str:
    if name == "iter":
        return "Iteration"
    return name


def make_legend_label(name: str) -> str:
    mapping = {
        "entropy_bits": "Entropy (bits)",
        "entropy_nats": "Entropy (nats)",
        "distinct_1": "Distinct-1",
        "distinct_2": "Distinct-2",
        "distinct_3": "Distinct-3",
        "m_real": "Real blocks",
        "m_synth": "Synthetic blocks",
        "test_ppl": "Test Perplexity",
        "log_test_ppl": "log(Test Perplexity)",
        "train_wall_sec": "Train wall time",
        "eval_wall_sec": "Eval wall time",
        "sample_wall_sec": "Sample wall time",
        "iter_wall_sec": "Iteration wall time",
        "total_gen_sec": "Total generation seconds",
        "approx_blocks_per_sec": "Blocks/sec",
        "approx_tokens_per_sec": "Tokens/sec",
    }

    return mapping.get(name, name)


def extract_alpha_value(run_name: str):
    """
    Extract numeric alpha from a run name like:
      alpha=0.1_llm_recursive_mix

    Returns float if found, else None.
    """
    if "_" in run_name:
        base = run_name.split("_")[0]
    else:
        base = run_name

    if base.startswith("alpha="):
        try:
            return float(base.split("=", 1)[1])
        except ValueError:
            return None

    return None


def pretty_run_label(run_name: str) -> str:
    """
    Convert:
      alpha=0.5_llm_recursive_mix -> r"$\\alpha=0.5$"

    while preserving the alpha=0.0 style formatting.
    """
    if "_" in run_name:
        base = run_name.split("_")[0]
    else:
        base = run_name

    alpha_val = extract_alpha_value(run_name)

    if alpha_val is not None:
        return rf"$\alpha={alpha_val:.1f}$"

    return base


def sanitize_ylim_for_scale(ylim, log_scale: bool):
    if ylim is None:
        return None

    lower, upper = ylim

    if log_scale and lower is not None and lower <= 0:
        print(f"Invalid ylim {ylim} for log scale. Adjusting lower bound to a small positive value.")
        lower = 1e-3

    return lower, upper


def plot_line(
    df: pd.DataFrame,
    x: str,
    y: str,
    title: str,
    outpath: Path,
    ylabel: str = "",
    global_dir: Path = None,
    run_name: str = None,
    log_scale: bool = False,
    ylim=None,
):
    if y not in df.columns:
        print(f"Skipping {y} because it is not present.")
        return

    plot_df = df[[x, y]].copy()
    plot_df[y] = pd.to_numeric(plot_df[y], errors="coerce")

    if log_scale:
        plot_df.loc[plot_df[y] <= 0, y] = float("nan")

        if plot_df[y].dropna().empty:
            print(f"Skipping {y} with log scale because there are no positive values.")
            return

    plt.figure()
    plt.plot(plot_df[x], plot_df[y], marker="o", linewidth=1)

    plt.xlabel(make_figure_label(x))
    plt.ylabel(ylabel if ylabel else make_legend_label(y))

    # Removed individual plot title
    # plt.title(title)

    if log_scale:
        plt.yscale("log")

    ylim = sanitize_ylim_for_scale(ylim, log_scale)

    if ylim is not None:
        plt.ylim(ylim)

    format_y_axis_1_decimal(log_scale=log_scale)

    savefig(outpath, global_dir=global_dir, run_name=run_name)


def plot_multi(
    df: pd.DataFrame,
    x: str,
    ys,
    title: str,
    outpath: Path,
    ylabel: str = "",
    global_dir: Path = None,
    run_name: str = None,
    log_scale: bool = False,
    ylim=None,
):
    ys_present = [y for y in ys if y in df.columns]

    if not ys_present:
        print(f"Skipping {title} because none of {ys} are present.")
        return

    plt.figure()
    plotted_any = False

    for y in ys_present:
        yvals = pd.to_numeric(df[y], errors="coerce").copy()

        if log_scale:
            yvals[yvals <= 0] = float("nan")

            if yvals.dropna().empty:
                print(f"Skipping {y} in {title} with log scale because there are no positive values.")
                continue

        plt.plot(
            df[x],
            yvals,
            marker="o",
            linewidth=1,
            label=make_legend_label(y),
        )

        plotted_any = True

    if not plotted_any:
        plt.close()
        print(f"Skipping {title} because nothing is plottable after log filtering.")
        return

    plt.xlabel(make_figure_label(x))
    plt.ylabel(ylabel if ylabel else "")

    # Removed individual plot title
    # plt.title(title)

    plt.legend()

    if log_scale:
        plt.yscale("log")

    ylim = sanitize_ylim_for_scale(ylim, log_scale)

    if ylim is not None:
        plt.ylim(ylim)

    format_y_axis_1_decimal(log_scale=log_scale)

    savefig(outpath, global_dir=global_dir, run_name=run_name)


def plot_compare_runs(
    run_dfs: dict,
    metric: str,
    title: str,
    outpath: Path,
    ylabel: str = "",
    x: str = "iter",
    log_scale: bool = False,
    ylim=None,
    show_y: bool = True,
    show_legend: bool = False,
):
    plotted_any = False

    plt.figure()

    # Sort runs numerically by alpha when possible
    sorted_items = sorted(
        run_dfs.items(),
        key=lambda kv: (
            extract_alpha_value(kv[0]) is None,
            float("inf") if extract_alpha_value(kv[0]) is None else extract_alpha_value(kv[0]),
            kv[0],
        ),
    )

    for run_name, df in sorted_items:
        if metric not in df.columns:
            print(f"Skipping {metric} for {run_name} because it is not present.")
            continue

        if x not in df.columns:
            print(f"Skipping {run_name}: missing x column {x}.")
            continue

        yvals = pd.to_numeric(df[metric], errors="coerce").copy()

        if log_scale:
            yvals[yvals <= 0] = float("nan")

            if yvals.dropna().empty:
                print(f"Skipping {metric} for {run_name} with log scale because there are no positive values.")
                continue

        plt.plot(
            df[x],
            yvals,
            marker="o",
            linewidth=1,
            label=pretty_run_label(run_name),
        )

        plotted_any = True

    if not plotted_any:
        plt.close()
        print(f"Skipping comparison plot {title} because metric {metric} is missing in all runs.")
        return

    plt.xlabel(make_figure_label(x))

    if show_y:
        plt.ylabel(ylabel if ylabel else make_legend_label(metric))

    # Removed comparison plot title
    # plt.title(title)

    if show_legend:
        plt.legend(title="Legend")

    if log_scale:
        plt.yscale("log")

    ylim = sanitize_ylim_for_scale(ylim, log_scale)

    if ylim is not None:
        plt.ylim(ylim)

    format_y_axis_1_decimal(log_scale=log_scale)

    savefig_single(outpath)


def add_log_perplexity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "test_ppl" in df.columns:
        df["log_test_ppl"] = df["test_ppl"].apply(
            lambda v: math.log(v)
            if isinstance(v, (int, float)) and pd.notna(v) and v > 0
            else float("nan")
        )

    return df


def main():
    # --------------------------------------------------
    # Define the runs you want to compare
    # Edit this list as needed
    # --------------------------------------------------
    run_dirs = [
        Path("runs/alpha=0.0_llm_recursive_mix"),
        Path("runs/alpha=0.1_llm_recursive_mix"),
        Path("runs/alpha=1.0_llm_recursive_mix"),
    ]

    # --------------------------------------------------
    # Optional log scaling
    # Put metric names here if you want them on log scale
    # --------------------------------------------------
    log_metrics = {
        # "test_ppl",
        # "entropy_bits",
        # "distinct_1",
        # "distinct_2",
        # "distinct_3",
        # "m_real",
        # "m_synth",
        # "train_wall_sec",
        # "eval_wall_sec",
        # "sample_wall_sec",
        # "iter_wall_sec",
        # "total_gen_sec",
        # "approx_blocks_per_sec",
        # "approx_tokens_per_sec",
    }

    global_plots_dir = Path("plots_global")
    compare_plots_dir = global_plots_dir / "comparisons"

    global_plots_dir.mkdir(parents=True, exist_ok=True)
    compare_plots_dir.mkdir(parents=True, exist_ok=True)

    run_dfs = {}

    # --------------------------------------------------
    # Existing behavior:
    # Save each run's plots locally and also copy them
    # into plots_global with prefixed names.
    # --------------------------------------------------
    for run_dir in run_dirs:
        run_name = run_dir.name
        plots_dir = run_dir / "plots"
        plots_dir.mkdir(parents=True, exist_ok=True)

        hist_path = run_dir / "history.jsonl"

        if not hist_path.exists():
            print(f"Skipping {run_name}: missing {hist_path}")
            continue

        df = read_history_jsonl(hist_path)
        df = add_log_perplexity(df)
        run_dfs[run_name] = df

        # Save CSV locally and globally
        df.to_csv(plots_dir / "history.csv", index=False)
        df.to_csv(global_plots_dir / f"{run_name}_history.csv", index=False)

        # --- Perplexity ---
        plot_line(
            df,
            "iter",
            "test_ppl",
            "Test Perplexity",
            plots_dir / "plot_test_ppl.png",
            ylabel="Perplexity",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("test_ppl" in log_metrics),
        )

        # --- log perplexity ---
        plot_line(
            df,
            "iter",
            "log_test_ppl",
            "log(Test Perplexity)",
            plots_dir / "plot_log_test_ppl.png",
            ylabel="log(Perplexity)",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=False,
        )

        # --- Entropy ---
        plot_multi(
            df,
            "iter",
            ["entropy_bits"],
            "Predictive Entropy",
            plots_dir / "plot_entropy.png",
            ylabel="Entropy",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("entropy_bits" in log_metrics),
        )

        # --- Distinct-n ---
        plot_multi(
            df,
            "iter",
            ["distinct_1", "distinct_2", "distinct_3"],
            "Distinct-n",
            plots_dir / "plot_distinct_n.png",
            ylabel="Distinct-n",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=any(
                m in log_metrics
                for m in ["distinct_1", "distinct_2", "distinct_3"]
            ),
            ylim=(0, 1),
        )

        # --- Dataset composition ---
        plot_multi(
            df,
            "iter",
            ["m_real", "m_synth"],
            "Real vs synthetic blocks per iteration",
            run_dir / "plot_real_vs_synth.png",
            ylabel="blocks",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=any(
                m in log_metrics
                for m in ["m_real", "m_synth"]
            ),
        )

        # --- Timing ---
        plot_multi(
            df,
            "iter",
            ["train_wall_sec", "eval_wall_sec", "sample_wall_sec", "iter_wall_sec"],
            "Wall-clock timing vs iteration",
            plots_dir / "plot_wall_times.png",
            ylabel="seconds",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=any(
                m in log_metrics
                for m in [
                    "train_wall_sec",
                    "eval_wall_sec",
                    "sample_wall_sec",
                    "iter_wall_sec",
                ]
            ),
        )

        # --- Synthetic timing fields ---
        plot_line(
            df,
            "iter",
            "total_gen_sec",
            "Synthetic generation total_gen_sec vs iteration",
            plots_dir / "plot_synth_total_gen_sec.png",
            ylabel="seconds",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("total_gen_sec" in log_metrics),
        )

        plot_line(
            df,
            "iter",
            "approx_blocks_per_sec",
            "Synthetic generation throughput",
            plots_dir / "plot_synth_blocks_per_sec.png",
            ylabel="blocks/sec",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("approx_blocks_per_sec" in log_metrics),
        )

        plot_line(
            df,
            "iter",
            "approx_tokens_per_sec",
            "Synthetic generation throughput",
            plots_dir / "plot_synth_tokens_per_sec.png",
            ylabel="tokens/sec",
            global_dir=global_plots_dir,
            run_name=run_name,
            log_scale=("approx_tokens_per_sec" in log_metrics),
        )

        print(f"Saved per-run plots for: {run_name}")

    # --------------------------------------------------
    # Comparison plots across runs
    # Saved into plots_global/comparisons/
    # --------------------------------------------------
    if not run_dfs:
        print("No valid runs found. No comparison plots created.")
        return

    comparison_specs = [
        ("test_ppl", "Test Perplexity", "compare_test_ppl.png", "Perplexity"),
        ("log_test_ppl", "log(Test Perplexity)", "compare_log_test_ppl.png", r"$\log(\mathrm{Perplexity})$"),
        ("entropy_bits", "Entropy (bits)", "compare_entropy_bits.png", "Bits"),
        ("distinct_1", "Distinct-1", "compare_distinct_1.png", "Proportion"),
        ("distinct_2", "Distinct-2", "compare_distinct_2.png", "Proportion"),
        ("distinct_3", "Distinct-3", "compare_distinct_3.png", "Proportion"),
        ("m_real", "Real Blocks vs Iteration Across Runs", "compare_m_real.png", "blocks"),
        ("m_synth", "Synthetic Blocks vs Iteration Across Runs", "compare_m_synth.png", "blocks"),
        ("train_wall_sec", "Train Wall Time vs Iteration Across Runs", "compare_train_wall_sec.png", "seconds"),
        ("eval_wall_sec", "Eval Wall Time vs Iteration Across Runs", "compare_eval_wall_sec.png", "seconds"),
        ("sample_wall_sec", "Sample Wall Time vs Iteration Across Runs", "compare_sample_wall_sec.png", "seconds"),
        ("iter_wall_sec", "Iteration Wall Time vs Iteration Across Runs", "compare_iter_wall_sec.png", "seconds"),
        ("total_gen_sec", "Total Generation Time vs Iteration Across Runs", "compare_total_gen_sec.png", "seconds"),
        ("approx_blocks_per_sec", "Generation Throughput Across Runs", "compare_blocks_per_sec.png", "blocks/sec"),
        ("approx_tokens_per_sec", "Generation Throughput Across Runs", "compare_tokens_per_sec.png", "tokens/sec"),
    ]

    for metric, title, filename, ylabel in comparison_specs:
        if metric in {"distinct_1", "distinct_2", "distinct_3"}:
            ylim = (0, 1)
        elif metric in {"entropy_bits"}:
            ylim = (1, None)
        elif metric in {"log_test_ppl"}:
            ylim = (2.5, None)
        else:
            ylim = None

        plot_compare_runs(
            run_dfs=run_dfs,
            metric=metric,
            title=title,
            outpath=compare_plots_dir / filename,
            ylabel=ylabel,
            log_scale=(metric in log_metrics),
            ylim=ylim,
        )

    print("Saved global copies to:", global_plots_dir.resolve())
    print("Saved comparison plots to:", compare_plots_dir.resolve())


if __name__ == "__main__":
    main()